# Subset sensitivity

In [9]:
import pandas as pd
import numpy as np
from monte import Monte, train_with_cv

## Load data

In [ ]:
df_meta_train = pd.read_csv("../../data/cancer-methyl/final_split/train_pan-cancer_meta.csv")
df_beta_train = pd.read_parquet("../../data/cancer-methyl/final_split/train_pan-cancer_beta.parquet")
df_meta_val = pd.read_csv("../../data/cancer-methyl/final_split/val_pan-cancer_meta.csv")
df_beta_val = pd.read_parquet("../../data/cancer-methyl/final_split/val_pan-cancer_beta.parquet")

In [3]:
df_beta = pd.concat([df_beta_train, df_beta_val])

df_meta = pd.concat([df_meta_train, df_meta_val])
df_meta = df_meta.set_index("Barcode", drop=False)
df_meta = df_meta.loc[df_beta.index]

we used the ESTIMATE as the primary metric for learning the cancer purity

In [6]:
df_meta_estimate = df_meta.dropna(subset=["ESTIMATE"])
df_beta_estimate = df_beta.loc[df_meta_estimate["Barcode"]]

**Test set**

In [ ]:
df_beta_test = pd.read_parquet("../../data/cancer-methyl/final_split/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/cancer-methyl/final_split/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

# remove samples with missing purity estimates
df_meta_test = df_meta_test.dropna(subset=["ESTIMATE"])
df_beta_test = df_beta_test.loc[df_meta_test["Barcode"]]

## Model training with subset

In [18]:
subset_sizes = [50, 100, 500, 1000, 5000, 10000, 50000, 100000]
n_repeats = 10

In [21]:
def evaluate_predictions(y_true, y_pred):
    from sklearn.metrics import mean_squared_error

    mse = mean_squared_error(y_true, y_pred)
    r2 = np.corrcoef(y_true, y_pred)[0, 1]

    return {"MSE": mse, "correlation": r2}

df_results = []
for subset_size in subset_sizes:
    for repeat in range(n_repeats):
        np.random.seed(repeat)
        subset_probes = np.random.choice(df_beta_estimate.columns, size=subset_size, replace=False)
        df_beta_subset = df_beta_estimate[subset_probes]

        monte = train_with_cv(df_beta_subset, df_meta_estimate["ESTIMATE"])

        # predict on training and test sets
        pred_train = monte.predict_purity(df_beta_subset)
        pred_test = monte.predict_purity(df_beta_test[subset_probes])

        # evaluation
        eval_train = evaluate_predictions(df_meta_estimate["ESTIMATE"], pred_train)
        eval_test = evaluate_predictions(df_meta_test["ESTIMATE"], pred_test)
        df_results.append({
            "subset_size": subset_size,
            "repeat": repeat,
            "train_MSE": eval_train["MSE"],
            "train_correlation": eval_train["correlation"],
            "test_MSE": eval_test["MSE"],
            "test_correlation": eval_test["correlation"]
        })

df_results = pd.DataFrame(df_results)

In [23]:
df_results

,subset_size,repeat,train_MSE,train_correlation,test_MSE,test_correlation
0,50,0,0.064600,0.340081,0.063492,0.331069
1,50,1,0.036072,0.485972,0.035578,0.480666
2,50,2,0.036786,0.488845,0.036055,0.499429
3,50,3,0.062392,0.310542,0.061096,0.345133
4,50,4,0.044900,0.440900,0.043357,0.445644
...,...,...,...,...,...,...
75,100000,5,0.010606,0.781906,0.010551,0.786807
76,100000,6,0.010952,0.773236,0.010798,0.778087
77,100000,7,0.009657,0.798013,0.009702,0.803477
78,100000,8,0.010827,0.776828,0.010618,0.784687


In [24]:
df_results.to_csv("../../data/monte_outputs/pancancer/pancancer_model_probe_subset_results.csv", index=False)